# MNIST Handwritten Digit Recognition Demo

This notebook demonstrates how to use the `mnist_recognition` package to train
and evaluate three neural network architectures on the MNIST dataset:

1. **Basic MLP** — Simple multi-layer perceptron
2. **Optimized Model** — Feature-selected efficient network
3. **Deep CNN** — Convolutional neural network for maximum accuracy

## Prerequisites

Install the package in editable mode from the repository root:

```bash
pip install -e .
```

In [ ]:
%matplotlib inline

import numpy as np
import tensorflow as tf

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

import mnist_recognition
print(f"Package version: {mnist_recognition.__version__}")

## 1. Data Loading and Exploration

In [ ]:
from mnist_recognition.data.loader import load_mnist, show_sample_images, explore_dataset

(x_train_raw, y_train_raw), (x_test_raw, y_test_raw) = load_mnist()
show_sample_images(x_train_raw, y_train_raw)
explore_dataset(x_train_raw, y_train_raw)

## 2. Data Preprocessing

In [ ]:
from mnist_recognition.data.preprocessing import preprocess_data
from mnist_recognition.data.augmentation import (
    create_data_augmentation,
    create_data_generators,
    visualize_augmentation,
)
from mnist_recognition.config import DEFAULT_CONFIG

config = DEFAULT_CONFIG.copy()

(x_train, y_train), (x_val, y_val), (x_test, y_test) = preprocess_data(
    x_train_raw, x_test_raw, y_train_raw, y_test_raw
)

# Augmentation preview
augmentation = create_data_augmentation()
visualize_augmentation(augmentation, x_train)

# Data generators
train_generator, val_generator = create_data_generators(
    x_train, y_train, x_val, y_val, config["batch_size"]
)

## 3. Model Training

### 3.1 Basic MLP Model

In [ ]:
from mnist_recognition.models.basic_mlp import create_and_train_basic_model

basic_model, basic_results = create_and_train_basic_model(
    config, train_generator, val_generator, x_val, y_val
)
print("\nBasic model training completed.")
basic_model.summary()

### 3.2 Optimized Model (Feature Selection)

In [ ]:
from mnist_recognition.models.optimized import create_and_train_optimized_model

optimized_model, reduced_data, optimized_results = create_and_train_optimized_model(
    config, x_train, y_train, x_val, y_val, x_test
)
x_train_reduced, x_val_reduced, x_test_reduced = reduced_data
print("\nOptimized model training completed.")
optimized_model.summary()

### 3.3 Deep CNN Model

In [ ]:
from mnist_recognition.models.deep_cnn import create_and_train_deep_model

deep_model, deep_results = create_and_train_deep_model(
    config, train_generator, val_generator
)
print("\nDeep CNN model training completed.")
deep_model.summary()

## 4. Training History

In [ ]:
from mnist_recognition.visualization.training import plot_training_history

results = {
    "basic_model": basic_results,
    "optimized_model": optimized_results,
    "deep_model": deep_results,
}

plot_training_history(results, ["basic_model", "optimized_model", "deep_model"])

## 5. Model Evaluation

In [ ]:
from mnist_recognition.evaluation.metrics import comprehensive_evaluation

results["basic_model"]["evaluation"] = comprehensive_evaluation(
    basic_model, x_test, y_test, "basic_model"
)
results["optimized_model"]["evaluation"] = comprehensive_evaluation(
    optimized_model, x_test_reduced, y_test, "optimized_model"
)
results["deep_model"]["evaluation"] = comprehensive_evaluation(
    deep_model, x_test, y_test, "deep_model"
)

## 6. Model Comparison

In [ ]:
from mnist_recognition.visualization.comparison import visualize_model_comparison

visualize_model_comparison(results)

## 7. Error Analysis

In [ ]:
from mnist_recognition.evaluation.error_analysis import analyze_errors

for name, model, test_x, opt in [
    ("basic_model", basic_model, x_test, False),
    ("optimized_model", optimized_model, x_test_reduced, True),
    ("deep_model", deep_model, x_test, False),
]:
    results[name]["error_analysis"] = analyze_errors(
        model, test_x, y_test, name, is_optimized=opt
    )

## 8. Sensitivity Analysis

In [ ]:
from mnist_recognition.visualization.sensitivity import perform_sensitivity_analysis

for name, model, test_x, opt in [
    ("basic_model", basic_model, x_test, False),
    ("optimized_model", optimized_model, x_test_reduced, True),
    ("deep_model", deep_model, x_test, False),
]:
    results[name]["sensitivity_report"] = perform_sensitivity_analysis(
        model, test_x, y_test, name, is_optimized=opt
    )

## 9. Final Report

In [ ]:
from mnist_recognition.evaluation.report import generate_final_report

comparison_df = generate_final_report(results, config)
comparison_df